In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_magres_new.magres') #latest magres file from 2025

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[ 1.85274192e+02  2.14597291e+00 -2.97030807e+00]
 [ 7.67493128e-02  1.76671585e+02 -5.39031136e+00]
 [-3.28009717e+00 -5.80118978e+00  1.90600456e+02]]

14N2 sigma:
 [[ 1.85274192e+02 -2.14597291e+00  2.97030807e+00]
 [-7.67493128e-02  1.76671585e+02 -5.39031136e+00]
 [ 3.28009717e+00 -5.80118978e+00  1.90600456e+02]]

14N3 sigma:
 [[1.85274192e+02 2.14597291e+00 2.97030807e+00]
 [7.67493128e-02 1.76671585e+02 5.39031136e+00]
 [3.28009717e+00 5.80118978e+00 1.90600456e+02]]

14N4 sigma:
 [[ 1.85274192e+02 -2.14597291e+00 -2.97030807e+00]
 [-7.67493128e-02  1.76671585e+02  5.39031136e+00]
 [-3.28009717e+00  5.80118978e+00  1.90600456e+02]]



In [6]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.3973307176178535

14N2 sigma:
 1.3973307176178624

14N3 sigma:
 1.3973307176178023

14N4 sigma:
 1.397330717617816



In [7]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 0                                      # atom for which parameters are wanted
CS_total[:,:] = atoms.species('N').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('N')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.0204 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.386 -0.26  -1.116]
 [-0.26  -0.349  0.285]
 [-1.116  0.285 -0.037]]

CS Tensor:
 [[ 1.85274e+02  2.14600e+00 -2.97000e+00]
 [ 7.70000e-02  1.76672e+02 -5.39000e+00]
 [-3.28000e+00 -5.80100e+00  1.90600e+02]]

CS isotropic Tensor:
 [[184.182   0.      0.   ]
 [  0.    184.182   0.   ]
 [  0.      0.    184.182]]

CS symmetric Tensor:
 [[185.274   1.111  -3.125]
 [  1.111 176.672  -5.596]
 [ -3.125  -5.596 190.6  ]]

CS antisymmetric Tensor:
 [[ 0.     1.035  0.155]
 [-1.035  0.     0.205]
 [-0.155 -0.205  0.   ]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.39459574 -0.96706553 -0.42753021] 

 Unsorted Eigenvectors:
 [[-0.74901044  0.62255697 -0.22672932]
 [ 0.21443189 -0.09600918 -0.97200885]
 [ 0.62689898  0.77666277  0.06158419]] 

Sorted Eigenvalues: 
 [-0.42753021 -0.96706553  1.39459574] 

Sorted Eigenvectors: 
 [[-0.22672932  0.62255697 -0.74901044]
 [-0.97200885 -0.09600918  0.21443189]
 [ 0.06158419  0.77666277  0.62689898]] 


For CS tensor
 Unsorted Eigenvalues:
 [193.85251334 183.99167945 174.70204098] 

 Unsorted Eigenvectors:
 [[-0.36064166 -0.93270383 -0.00107868]
 [-0.30985315  0.11871763  0.9433436 ]
 [ 0.87973213 -0.34054324  0.33181574]] 

Sorted Eigenvalues: 
 [183.99167945 174.70204098 193.85251334] 

Sorted Eigenvectors: 
 [[-0.93270383 -0.00107868 -0.36064166]
 [ 0.11871763  0.9433436  -0.30985315]
 [-0.34054324  0.33181574  0.87973213]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.4275302065467817 -0.9670655308765369 1.3945957374234002
CSA Tensor Components δyy, δxx, δzz: 
 183.99167945082232 174.70204097884823 193.8525133360463


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        1.3946
etaq            0.386876
iso_cs (ppm)  184.182
csa (ppm)       9.67044
etas            0.960623


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.62255697 -0.22672932 -0.74901044]
 [-0.09600918 -0.97200885  0.21443189]
 [ 0.77666277  0.06158419  0.62689898]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
4.5336881955405515 51.178296502810205 15.975724688151425 

Direction cosine csa: 

[[-0.00107868 -0.93270383 -0.36064166]
 [ 0.9433436   0.11871763 -0.30985315]
 [ 0.33181574 -0.34054324  0.87973213]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-45.74368092695918 28.389932788328547 -40.66824245972513 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 51.15963478777265 chi: 40.958450161434406 xi: -41.82414562896251 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[185.27419243   1.11136111  -3.12520262]
 [  1.11136111 176.67158518  -5.59575057]
 [ -3.12520262  -5.59575057 190.60045615]]
CSA Tensor in Tenon Frame: 
 [[178.727605    -6.91762369   3.58521645]
 [ -6.91762369 189.00650709  -2.07204457]
 [  3.58521645  -2.07204457 184.81212167]]
Quad Tensor in Crystal Frame: 
 [[ 0.38560118 -0.26040613 -1.11646071]
 [-0.26040613 -0.34872028  0.28517457]
 [-1.11646071  0.28517457 -0.03688091]]
Quad Tensor in Tenon Frame: 
 [[-0.4312279   0.08214015 -0.05392948]
 [ 0.08214015  1.37332846  0.20499056]
 [-0.05392948  0.20499056 -0.94210057]]
